# Exam summer 2026

In [2]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests


## 1 Real GDP across US states


#### 1.1.2 Common years and states

After keeping only the years available in both datasets and removing states with missing observations, we are left with 29 years and 50 states.

In [4]:

# 1.1.1 Import state codes and regions
from states import STATES, REGION

# FRED API key
api_key = '9829eaf99bf0bb5da6b84dfcda3fb0e4'

# Function for downloading one series from FRED
def download_fred_series(series_id):

    # FRED API URL
    url = 'https://api.stlouisfed.org/fred/series/observations'

    # Information sent to FRED
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json'
    }

    # Download the data
    response = requests.get(url, params=params)
    response.raise_for_status()

    # Convert the data to a dataframe
    data = pd.DataFrame(response.json()['observations'])

    # Convert date to year
    data['year'] = pd.to_datetime(data['date']).dt.year

    # Convert values to numbers
    data['value'] = pd.to_numeric(
        data['value'],
        errors='coerce'
    )

    # Return a series with year as index
    return data.set_index('year')['value']


# Empty dataframes
gdp = pd.DataFrame()
pop = pd.DataFrame()

# Download GDP and population for all 50 states
for state in STATES:

    # Real GDP
    gdp[state] = download_fred_series(
        f'{state}RGSP'
    )

    # Population
    pop[state] = download_fred_series(
        f'{state}POP'
    )


# Check the dataframes
#display(gdp.head())
#display(pop.head())

#print("GDP shape:", gdp.shape)
#print("Population shape:", pop.shape)


# 1.1.2 Keep common years and remove states with missing observations

# Find the years that are in both dataframes
common_years = gdp.index.intersection(pop.index)

# Keep only the common years
gdp_common = gdp.loc[common_years].copy()
pop_common = pop.loc[common_years].copy()

# Find states with no missing values in either dataframe
valid_states = (
    gdp_common.notna().all()
    & pop_common.notna().all()
)

# Keep only states without missing observations
gdp = gdp_common.loc[:, valid_states]
pop = pop_common.loc[:, valid_states]

# Report number of years and states
print("Number of years:", len(gdp.index))
print("Number of states:", len(gdp.columns))


# 1.1.3 Compute real GDP per person in dollars

# GDP is in millions and population is in thousands
y = (gdp / pop) * 1000

display(y.head())

# 1.1.4 Compare real GDP per person in the first and last year

# First and last year in the data
tfirst = y.index.min()
tlast = y.index.max()

# Loop over the first and last year
for year in [tfirst, tlast]:

    # GDP per person for all states in this year
    y_year = y.loc[year]

    # Highest and lowest state
    highest_state = y_year.idxmax()
    lowest_state = y_year.idxmin()

    # Highest and lowest GDP per person
    highest = y_year.max()
    lowest = y_year.min()

    # Ratio and average across states
    ratio = highest / lowest
    average = y_year.mean()

    # Report results
    print(f"Year: {year}")
    print(f"Highest: {highest_state}, ${highest:,.2f}")
    print(f"Lowest: {lowest_state}, ${lowest:,.2f}")
    print(f"Highest / lowest ratio: {ratio:.2f}")
    print(f"Average: ${average:,.2f}")
    print()


Number of years: 29
Number of states: 50


,AL,AK,AZ,AR,CA,CO,CT,DE,FL,GA,...,SD,TN,TX,UT,VT,VA,WA,WV,WI,WY
year,,,,,,,,,,,,,,,,,,,,,
1997,35807.856017,67457.123805,39605.821967,34540.435110,44733.982939,48684.486108,63014.935839,69748.742898,41134.250699,46911.500176,...,37102.024341,42174.514399,44434.405916,41775.745777,36306.218308,47526.867411,50234.997381,33247.961542,42878.542989,50533.611371
1998,36863.947606,65447.615023,42314.201621,35420.860909,47077.134837,52015.826788,64858.950003,75916.652555,42529.012498,49460.461189,...,39371.282272,44354.433604,46674.180385,43470.604533,37702.830612,49921.971948,52711.999932,34070.822349,44443.046546,51671.614120
1999,38109.166834,64218.079096,44847.009375,37139.571517,49945.163875,54495.451702,66684.318338,80773.895942,43950.848785,51932.811521,...,40972.511127,45309.731770,47756.958006,44701.469972,39683.531512,51854.555973,56063.474824,35248.609795,46096.274488,53933.886848
2000,37890.553669,61194.847467,43547.244441,35656.771403,52498.582072,55175.331373,68549.087470,80609.074828,43236.432557,50771.619813,...,42502.024227,44011.670280,47263.240816,43951.174915,40754.374051,52480.115939,55131.873516,35065.779534,46254.567595,53842.201092
2001,37704.140491,63142.679505,43782.422868,35491.800142,51757.417416,54827.668563,68846.332550,82600.078673,43690.131776,50463.135060,...,42741.552458,43526.444806,47867.499715,43895.275899,41948.440356,53685.088358,53103.134426,35304.174732,46781.860367,57288.383668


Year: 1997
Highest: DE, $69,748.74
Lowest: ID, $32,062.68
Highest / lowest ratio: 2.18
Average: $44,592.95

Year: 2025
Highest: NY, $94,701.99
Lowest: MS, $42,434.03
Highest / lowest ratio: 2.23
Average: $64,674.46

